# Satellite altimetry (IMOS, 1985-present) from the cloud

Along-track significant wave height (Hs) and wind speed of **18 altimeter missions**
(GEOSAT 1985 to Sentinel-6A and the SWOT nadir altimeter), calibrated against buoys
(Ribal and Young, 2019; IMOS DM00 archive), served by GeoOcean as a Zarr store.

The store has one dimension `obs`, sorted by 1x1 degree cell and time, and a cell index
(`cell_start`, `cell_count`), so the observations around a point are read as a few small
slices. `bluemath_tk.waves.altimetry` does it for you.

Requirements: `bluemath-tk` (with `bluemath_tk.waves.altimetry`), `xarray`, `zarr>=3`, `fsspec`, `aiohttp`, `matplotlib`, `cartopy`. The stores are read over HTTP: only the chunks needed are downloaded.

In [ ]:
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature

from bluemath_tk.core.metrics import compute_metrics
from bluemath_tk.waves.altimetry import open_altimeter, altimeter_near, altimeter_passes, collocate

# Public GeoOcean Zarr stores. On the GeoOcean cluster use "http://geocean01.sci.unican.es/zarr" (faster).
BASE = "https://geoocean.unican.es/zarr"
IMOS_URL = f"{BASE}/IMOS/imos_altimeter_dm00.zarr"
SWOT_URL = f"{BASE}/SWOT/swot_karin_windwave.zarr"
WHACS_URL = f"{BASE}/CSIRO/whacs_global_smc_1h_2020_2023.zarr"

BLUE, ORANGE, AQUA, GREY = "#2a78d6", "#eb6834", "#1baf7a", "#52514e"
plt.rcParams.update({"axes.grid": True, "grid.color": "#e4e3df", "axes.spines.top": False,
                     "axes.spines.right": False, "figure.dpi": 110})

LON, LAT = -3.8, 43.7  # Cantabrian Sea, ~20 km off Santander

## Open the store

Its attributes say what it is and how to use it.

In [ ]:
ds = open_altimeter(IMOS_URL)
for key in ("title", "summary", "usage", "time_coverage_start", "time_coverage_end", "license"):
    print(f"{key}: {ds.attrs[key]}\n")
ds

In [ ]:
satellites = np.array(ds["satellite"].attrs["flag_meanings"].split())
print(ds["satellite"].attrs["comment"])
satellites

## Observations around a point

All observations within 50 km of the point, in water deeper than 20 m, with good or
probably good quality flags (the default `flags=(1, 2)`).

In [ ]:
obs = altimeter_near(ds, LON, LAT, radius_km=50, min_depth=20)
print(f"{len(obs):,} observations, {obs.time.min():%Y-%m-%d} to {obs.time.max():%Y-%m-%d}")
obs.assign(satellite=satellites[obs.satellite]).head()

One year of tracks: each line is a satellite crossing the 50 km circle.

In [ ]:
year = obs[obs.time.dt.year == 2024]
fig = plt.figure(figsize=(7, 5.5))
ax = plt.axes(projection=ccrs.PlateCarree())
ax.set_extent([LON - 1.1, LON + 1.1, LAT - 0.75, LAT + 0.75])
ax.add_feature(cfeature.LAND, color="#e4e3df", zorder=0)
ax.coastlines("50m", lw=0.6)
sc = ax.scatter(year.lon, year.latitude, c=year.hs, s=5, cmap="viridis", vmin=0, vmax=6,
                transform=ccrs.PlateCarree())
ax.plot(LON, LAT, "*", color="k", ms=12, transform=ccrs.PlateCarree())
plt.colorbar(sc, label="Hs (m)", shrink=0.8)
ax.set_title(f"Altimeter tracks within 50 km, 2024 ({year.satellite.nunique()} missions)", loc="left")
plt.show()

## One value per pass

Successive 1 Hz values along a track are strongly correlated (and CFOSAT has 5 per
second), so before any statistics each crossing of the circle is reduced to its median.

In [ ]:
passes = altimeter_passes(obs)
passes["satellite"] = satellites[passes.satellite.astype(int)]
print(f"{len(passes):,} passes")

fig, ax = plt.subplots(figsize=(12, 3.5))
ax.plot(passes.index, passes.hs, ".", color=BLUE, ms=2, alpha=0.6)
ax.set_ylabel("Hs (m)")
ax.set_title(f"Altimeter Hs within 50 km of ({LON}, {LAT}), one value per pass", loc="left")
plt.show()

## Statistics

In [ ]:
monthly = passes.groupby(passes.index.month).hs.agg(median="median", p90=lambda x: x.quantile(0.9))
fig, ax = plt.subplots(figsize=(7, 3.5))
ax.plot(monthly.index, monthly["median"], "o-", color=BLUE, lw=2, label="median")
ax.plot(monthly.index, monthly["p90"], "o-", color=ORANGE, lw=2, label="90th percentile")
ax.set_xticks(range(1, 13), ["J", "F", "M", "A", "M", "J", "J", "A", "S", "O", "N", "D"])
ax.set_ylabel("Hs (m)")
ax.set_title("Monthly climatology of Hs (all missions, 1985-present)", loc="left")
ax.legend(frameon=False)
plt.show()

Passes and Hs per mission, in launch order: the record is made of many missions.

In [ ]:
per_mission = passes.groupby("satellite").agg(
    first=("hs", lambda x: x.index.min().year), last=("hs", lambda x: x.index.max().year),
    passes=("hs", "size"), hs_median=("hs", "median"), hs_p90=("hs", lambda x: x.quantile(0.9)))
per_mission.sort_values("first").round(2)

In [ ]:
annual = passes.hs.resample("YS").agg(["mean", lambda x: x.quantile(0.95), "size"])
annual.columns = ["mean", "p95", "passes"]
annual = annual[annual.passes >= 20]  # years with too few passes are left out
fig, ax = plt.subplots(figsize=(10, 3.5))
ax.plot(annual.index.year, annual["mean"], "o-", color=BLUE, lw=2, label="annual mean")
ax.plot(annual.index.year, annual["p95"], "o-", color=ORANGE, lw=2, label="annual 95th percentile")
ax.set_ylabel("Hs (m)")
ax.set_title("Annual Hs statistics (passes per year change with the missions in orbit)", loc="left")
ax.legend(frameon=False)
plt.show()

## Notes

- Keep the quality flags (`flag` 1-2) and reduce to one value per pass before statistics.
- Sampling changes with time: 1-3 missions before 2000, up to 9 after 2020.
- Near the coast the altimeter footprint degrades: `min_depth` and `min_dist2coast` help.
- Cite Ribal and Young (2019) and acknowledge IMOS (see the store `license`).